In [1]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.ops import unary_union

from pathlib import Path
from tqdm.auto import tqdm

datasets = Path("/nas/cee-ice/data")

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subbasins = gpd.read_parquet(metadata_dir / 'subbasins.parquet')
subbasins.index = subbasins.index.astype(str)
subbasins.geometry = subbasins.geometry.centroid
footprint_geom = unary_union(subbasins.geometry.envelope)

In [5]:
# Read in hydroatlas
gdb_path = datasets / "HydroSHEDS" / "HydroATLAS" / "BasinATLAS_v10.gdb"
layer_name = "BasinATLAS_v10_lev09"

hydroatlas = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio", use_arrow=True)
hydroatlas.replace({'wet_cl_smj':-9999}, 13, inplace=True)
hydroatlas = hydroatlas.set_index('HYBAS_ID')

# hydroatlas_filt = hydroatlas[hydroatlas.intersects(footprint_geom)]
# hydroatlas_filt

/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/pyogrio/raw.py:337: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  table = reader.read_all()


In [ ]:
hydroatlas_filt = hydroatlas[hydroatlas.intersects(footprint_geom)]
hydroatlas_filt

In [10]:
hydroatlas_filt = hydroatlas

In [14]:
hybas_pairs.index

Index(['23021321', '23021261', '23021244', '23021122', '23021112', '23021112',
       '23021045', '23018471', '23014554', '23014554',
       ...
       'USGS-15908000', 'USGS-15908000', 'USGS-15908000', 'USGS-15908000',
       'USGS-15908000', 'USGS-15905100', 'USGS-15905100', 'USGS-15905100',
       'USGS-15905100', 'USGS-15905100'],
      dtype='object', name='site_id', length=112790)

In [15]:
subs_buff = subbasins.copy()
subs_buff = subs_buff.to_crs("EPSG:6933")
subs_buff.geometry = subs_buff.buffer(2500)

hybas_pairs = gpd.sjoin(
    subs_buff,
    hydroatlas_filt.to_crs("EPSG:6933"),
    how='left',
)

hybas_area = hybas_pairs['UP_AREA']
merit_area = hybas_pairs['uparea_km2']
hybas_pairs['hybas_area_diff'] = (hybas_area - merit_area) / ((hybas_area + merit_area)/2)

def get_best_hybas(grp):
     # Pick best match by area.
    idx_min = grp['hybas_area_diff'].abs().idxmin()
    return grp.loc[idx_min]

all_attributes = hybas_pairs.reset_index().groupby('site_id').apply(get_best_hybas, include_groups=False)

In [17]:
all_attributes

,geometry,area_km2,uparea_km2,node_type,is_gauge,nextdown,outlet_id,HYBAS_ID,NEXT_DOWN,NEXT_SINK,...,hft_ix_s09,hft_ix_u09,gad_id_smj,gdp_ud_sav,gdp_ud_ssu,gdp_ud_usu,hdi_ix_sav,Shape_Length,Shape_Area,hybas_area_diff
site_id,,,,,,,,,,,,,,,,,,,,,
21000002,"POLYGON ((562172.019 5410060.48, 562159.981 54...",468.102025,4364.328752,merged,False,EAUF-U0820010,EAUF-V7200010,2.090493e+09,2.090494e+09,2.090017e+09,...,106,110,79,29174,3.976704e+08,4.308199e+09,910,1.546063,0.078265,-0.059877
21000020,"POLYGON ((591434.06 5415004.691, 591422.022 54...",401.905725,3625.201976,merged,False,EAUF-U0610010,EAUF-V7200010,2.090490e+09,2.090493e+09,2.090017e+09,...,163,104,79,29174,1.899765e+08,2.767089e+09,910,0.499102,0.010225,-0.164848
21000022,"POLYGON ((586561.776 5427439.266, 586549.738 5...",292.309508,3076.492166,merged,False,21000020,EAUF-V7200010,2.090488e+09,2.090490e+09,2.090017e+09,...,88,92,79,29174,1.372329e+08,1.245171e+09,910,0.765264,0.023484,-0.440938
21000026,"POLYGON ((558935.353 5432274.463, 558923.315 5...",481.039377,1767.807919,merged,False,21000022,EAUF-V7200010,2.090486e+09,2.090486e+09,2.090017e+09,...,92,92,79,30484,2.234216e+08,2.234215e+08,899,1.265614,0.054663,-1.183507
21000032,"POLYGON ((565028.51 5444560.247, 565016.472 54...",390.920772,621.770285,merged,False,EAUF-U0230010,EAUF-V7200010,2.090484e+09,2.090486e+09,2.090017e+09,...,86,86,79,28545,2.993196e+08,2.993199e+08,902,1.646779,0.076334,0.015052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-50129254,"POLYGON ((-6451841.411 2267345.611, -6451853.4...",42.674493,42.674493,gauge,True,None,USGS-50129254,7.090073e+09,0.000000e+00,7.090073e+09,...,277,277,181,34887,1.574034e+09,1.574035e+09,832,0.780912,0.018343,1.339595
USGS-50136400,"POLYGON ((-6464497.679 2281391.831, -6464509.7...",51.402873,51.402873,gauge,True,USGS-50138000,USGS-50138000,7.090073e+09,0.000000e+00,7.090073e+09,...,346,376,181,34887,1.705076e+09,6.534238e+08,832,0.535892,0.005852,0.288039
USGS-50138000,"POLYGON ((-6465733.733 2272749.631, -6465745.7...",261.453879,312.856752,gauge,True,None,USGS-50138000,7.090073e+09,0.000000e+00,7.090073e+09,...,288,288,181,34887,3.784680e+09,3.784679e+09,832,0.901285,0.029002,0.085791


In [20]:
all_attributes

,geometry,area_km2,uparea_km2,node_type,is_gauge,nextdown,outlet_id,HYBAS_ID,NEXT_DOWN,NEXT_SINK,...,hft_ix_s09,hft_ix_u09,gad_id_smj,gdp_ud_sav,gdp_ud_ssu,gdp_ud_usu,hdi_ix_sav,Shape_Length,Shape_Area,hybas_area_diff
site_id,,,,,,,,,,,,,,,,,,,,,
21000002,"POLYGON ((562172.019 5410060.48, 562159.981 54...",468.102025,4364.328752,merged,False,EAUF-U0820010,EAUF-V7200010,2.090493e+09,2.090494e+09,2.090017e+09,...,106,110,79,29174,3.976704e+08,4.308199e+09,910,1.546063,0.078265,-0.059877
21000020,"POLYGON ((591434.06 5415004.691, 591422.022 54...",401.905725,3625.201976,merged,False,EAUF-U0610010,EAUF-V7200010,2.090490e+09,2.090493e+09,2.090017e+09,...,163,104,79,29174,1.899765e+08,2.767089e+09,910,0.499102,0.010225,-0.164848
21000022,"POLYGON ((586561.776 5427439.266, 586549.738 5...",292.309508,3076.492166,merged,False,21000020,EAUF-V7200010,2.090488e+09,2.090490e+09,2.090017e+09,...,88,92,79,29174,1.372329e+08,1.245171e+09,910,0.765264,0.023484,-0.440938
21000026,"POLYGON ((558935.353 5432274.463, 558923.315 5...",481.039377,1767.807919,merged,False,21000022,EAUF-V7200010,2.090486e+09,2.090486e+09,2.090017e+09,...,92,92,79,30484,2.234216e+08,2.234215e+08,899,1.265614,0.054663,-1.183507
21000032,"POLYGON ((565028.51 5444560.247, 565016.472 54...",390.920772,621.770285,merged,False,EAUF-U0230010,EAUF-V7200010,2.090484e+09,2.090486e+09,2.090017e+09,...,86,86,79,28545,2.993196e+08,2.993199e+08,902,1.646779,0.076334,0.015052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-50129254,"POLYGON ((-6451841.411 2267345.611, -6451853.4...",42.674493,42.674493,gauge,True,None,USGS-50129254,7.090073e+09,0.000000e+00,7.090073e+09,...,277,277,181,34887,1.574034e+09,1.574035e+09,832,0.780912,0.018343,1.339595
USGS-50136400,"POLYGON ((-6464497.679 2281391.831, -6464509.7...",51.402873,51.402873,gauge,True,USGS-50138000,USGS-50138000,7.090073e+09,0.000000e+00,7.090073e+09,...,346,376,181,34887,1.705076e+09,6.534238e+08,832,0.535892,0.005852,0.288039
USGS-50138000,"POLYGON ((-6465733.733 2272749.631, -6465745.7...",261.453879,312.856752,gauge,True,None,USGS-50138000,7.090073e+09,0.000000e+00,7.090073e+09,...,288,288,181,34887,3.784680e+09,3.784679e+09,832,0.901285,0.029002,0.085791


In [26]:
attributes_dir = save_dir / 'attributes'

In [30]:
hybas_properties = pd.read_csv(attributes_dir / "properties.txt", header=None)[0].to_list()
matchup_props = ['outlet_id', 'area_km2','uparea_km2']
all_props = matchup_props + hybas_properties + ['hybas_area_diff']

In [34]:
attributes = (
    all_attributes[all_props]
    .reset_index()
    .rename(columns={'site_id':'subbasin', 'outlet_id': 'basin'})
    .astype({'subbasin':str, 'basin': str})
    .set_index(['basin', 'subbasin'])
)
attributes

area_km2   uparea_km2  dis_m3_pyr  dis_m3_pmn  \
basin         subbasin                                                         
EAUF-V7200010 21000002       468.102025  4364.328752   66.342003   22.684999   
              21000020       401.905725  3625.201976   48.854000   16.743000   
              21000022       292.309508  3076.492166   28.339001    9.780000   
              21000026       481.039377  1767.807919    6.031000    2.077000   
              21000032       390.920772   621.770285    8.653000    3.024000   
...                                 ...          ...         ...         ...   
USGS-50129254 USGS-50129254   42.674493    42.674493    4.198000    1.576000   
USGS-50138000 USGS-50136400   51.402873    51.402873    0.185000    0.068000   
              USGS-50138000  261.453879   312.856752    5.788000    2.068000   
USGS-50144000 USGS-50144000  349.557308   349.557308    9.208000    3.183000   
USGS-50147800 USGS-50147800  195.872993   195.872993    5.115000    1.855000   

                             dis_m3_pmx  run_mm_syr  inu_pc_smn  inu_pc_smx  \
basin         subbasin                                                        
EAUF-V7200010 21000002       108.996002         525           0           0   
              21000020        80.602997         537           0           0   
              21000022        46.625000         504           0           0   
              21000026         9.826000         422           0           0   
              21000032        14.126000         437           0           0   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254    8.597000         578          61          69   
USGS-50138000 USGS-50136400    0.359000         598          63          79   
              USGS-50138000   11.577000         593          30          48   
USGS-50144000 USGS-50144000   19.839001         609          36          61   
USGS-50147800 USGS-50147800   10.214000         601          26          38   

                             inu_pc_slt  lka_pc_sse  ...  urb_pc_sse  \
basin         subbasin                               ...               
EAUF-V7200010 21000002                1           0  ...           0   
              21000020                0           0  ...           0   
              21000022                0           1  ...           0   
              21000026                0           0  ...           0   
              21000032                0           0  ...           0   
...                                 ...         ...  ...         ...   
USGS-50129254 USGS-50129254          72           1  ...          22   
USGS-50138000 USGS-50136400          80           0  ...          48   
              USGS-50138000          51           0  ...          27   
USGS-50144000 USGS-50144000          62           1  ...           4   
USGS-50147800 USGS-50147800          38           0  ...          39   

                             nli_ix_sav  rdd_mk_sav  hft_ix_s93  hft_ix_s09  \
basin         subbasin                                                        
EAUF-V7200010 21000002              844         711         135         106   
              21000020             1754         786         187         163   
              21000022              842         631         133          88   
              21000026              728         759         142          92   
              21000032              824         536         127          86   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254        3209        2365         296         277   
USGS-50138000 USGS-50136400        4778        5447         363         346   
              USGS-50138000        3680        2432         291         288   
USGS-50144000 USGS-50144000        1686        1292         222         173   
USGS-50147800 USGS-50147800        3856        1703         301         286  

In [35]:
attributes.to_parquet(attributes_dir / f"static.parquet")

In [36]:
test = pd.read_parquet(attributes_dir / "static.parquet")

In [37]:
test

area_km2   uparea_km2  dis_m3_pyr  dis_m3_pmn  \
basin         subbasin                                                         
EAUF-V7200010 21000002       468.102025  4364.328752   66.342003   22.684999   
              21000020       401.905725  3625.201976   48.854000   16.743000   
              21000022       292.309508  3076.492166   28.339001    9.780000   
              21000026       481.039377  1767.807919    6.031000    2.077000   
              21000032       390.920772   621.770285    8.653000    3.024000   
...                                 ...          ...         ...         ...   
USGS-50129254 USGS-50129254   42.674493    42.674493    4.198000    1.576000   
USGS-50138000 USGS-50136400   51.402873    51.402873    0.185000    0.068000   
              USGS-50138000  261.453879   312.856752    5.788000    2.068000   
USGS-50144000 USGS-50144000  349.557308   349.557308    9.208000    3.183000   
USGS-50147800 USGS-50147800  195.872993   195.872993    5.115000    1.855000   

                             dis_m3_pmx  run_mm_syr  inu_pc_smn  inu_pc_smx  \
basin         subbasin                                                        
EAUF-V7200010 21000002       108.996002         525           0           0   
              21000020        80.602997         537           0           0   
              21000022        46.625000         504           0           0   
              21000026         9.826000         422           0           0   
              21000032        14.126000         437           0           0   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254    8.597000         578          61          69   
USGS-50138000 USGS-50136400    0.359000         598          63          79   
              USGS-50138000   11.577000         593          30          48   
USGS-50144000 USGS-50144000   19.839001         609          36          61   
USGS-50147800 USGS-50147800   10.214000         601          26          38   

                             inu_pc_slt  lka_pc_sse  ...  urb_pc_sse  \
basin         subbasin                               ...               
EAUF-V7200010 21000002                1           0  ...           0   
              21000020                0           0  ...           0   
              21000022                0           1  ...           0   
              21000026                0           0  ...           0   
              21000032                0           0  ...           0   
...                                 ...         ...  ...         ...   
USGS-50129254 USGS-50129254          72           1  ...          22   
USGS-50138000 USGS-50136400          80           0  ...          48   
              USGS-50138000          51           0  ...          27   
USGS-50144000 USGS-50144000          62           1  ...           4   
USGS-50147800 USGS-50147800          38           0  ...          39   

                             nli_ix_sav  rdd_mk_sav  hft_ix_s93  hft_ix_s09  \
basin         subbasin                                                        
EAUF-V7200010 21000002              844         711         135         106   
              21000020             1754         786         187         163   
              21000022              842         631         133          88   
              21000026              728         759         142          92   
              21000032              824         536         127          86   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254        3209        2365         296         277   
USGS-50138000 USGS-50136400        4778        5447         363         346   
              USGS-50138000        3680        2432         291         288   
USGS-50144000 USGS-50144000        1686        1292         222         173   
USGS-50147800 USGS-50147800        3856        1703         301         286  